# Price-Path Simulation in Python

This notebook simulates stock price paths using geometric Brownian motion. The simulated paths are not market predictions. They are controlled scenarios that will later be used to test delta hedging under many possible stock movements.

## Core concepts

A random walk is a path built from random moves. In finance, a stock path is often modeled with random returns instead of random price changes.

Geometric Brownian motion is a simple model where the stock price evolves through compounded random returns:

\[
S_{t+\Delta t}
=
S_t
\exp\left((\mu - 0.5\sigma^2)\Delta t + \sigma\sqrt{\Delta t}Z\right)
\]

Here:

| Symbol | Meaning |
|---|---|
| \(S_t\) | stock price at time \(t\) |
| \(\mu\) | drift, or average expected return assumption |
| \(\sigma\) | volatility |
| \(\Delta t\) | time step |
| \(Z\) | standard normal random shock |

Monte Carlo simulation means generating many paths instead of one path. A random seed makes the results reproducible.

## Why simulation helps hedging analysis

Delta hedging needs a stock path. A single historical path only shows one outcome. Simulation lets us test the hedge across many possible paths, then study the distribution of hedging errors.

This is still simplified. Geometric Brownian motion assumes constant drift and volatility, continuous paths, and normally distributed log returns. Real markets have jumps, changing volatility, liquidity shocks, and transaction costs.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.simulation import (
    build_simulation_figures,
    final_price_distribution,
    save_simulation_outputs,
    simulate_gbm_paths,
    simulation_summary,
)

## Simulation inputs

In [ ]:
starting_price = 100.0
drift = 0.06
volatility = 0.25
time_horizon = 30 / 365.25
steps = 30
num_paths = 2_000
random_seed = 42

## Simulate stock paths

In [ ]:
paths = simulate_gbm_paths(
    starting_price=starting_price,
    drift=drift,
    volatility=volatility,
    time_horizon=time_horizon,
    steps=steps,
    num_paths=num_paths,
    random_seed=random_seed,
)

paths.head()

## Final price distribution

In [ ]:
final_prices = final_price_distribution(paths)
final_prices.describe()

## Simulation summary table

In [ ]:
summary = simulation_summary(paths)
summary

## Save outputs

In [ ]:
processed_dir = project_root / "data" / "processed"
figures_dir = project_root / "figures"

saved_tables = save_simulation_outputs(
    paths,
    output_dir=processed_dir,
    file_prefix="gbm",
)

saved_figures = build_simulation_figures(
    paths,
    output_dir=figures_dir,
)

saved_tables, saved_figures

## Interpretation

The sample path plot shows individual simulated stock paths. The final price distribution shows where the stock ended across all simulated paths. Later, each path can be passed into the delta-hedging engine to measure hedge P&L and hedging error.